# DeepConvLSTM on DAGHAR — LODO (Colab)

Faithful PyTorch re-implementation of DeepConvLSTM (Ordonez & Roggen, 2016 — DAGHAR's "ConvNet [13]").

**run2 recipe** (architecture UNCHANGED — only preprocessing & training-recipe knobs):
- frequency view = `log1p(|FFT|)` magnitude with a **Hann** window
- per-feature z-score (fit on source-train only)
- model selection / early stopping on **validation accuracy** (not loss)
- max_epochs=100, patience=20

Results are written to `results/<run_name>/<view>/`, so run1 (your earlier results at `results/time` and `results/frequency`) is left untouched.

> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/Project_ConvNet_DAGHAR`; they are kept as the record of that run.
> Note: `run_lodo.py` / the P18b runners are **not** resumable — re-running a training cell retrains and rewrites that run's output folder.
>
> **Adopted recipe = `run2_nohann`** (frequency view, log-magnitude FFT, no Hann taper, DC kept, val-accuracy selection) — ablation C in the last cell. Only `results/run2_nohann/` is included in Final_PROJECT; the other runs (`run1`, `run2`, `run2_dropdc`, `run2_f1`) were exploratory and stay in the original phase folder. Re-running cells 3/4/6-A/6-B would create those folders fresh.

## 1. Mount Drive & set paths

In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("convnet_a")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
print('Datasets on disk:', sorted(os.listdir(DATA_PATH)))
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Smoke test (fast) — run2 frequency recipe on 2 datasets

In [2]:
!cd "$PROJECT_PATH" && python -m experiments.smoke_test --data_path "$DATA_PATH" --datasets UCI,WISDM --target WISDM --input_view frequency

device=cuda  target=WISDM  datasets=['UCI', 'WISDM']
meta: {'target': 'WISDM', 'sources': ['UCI'], 'input_view': 'frequency', 'seq_len': 31, 'normalize': 'zscore_feature', 'fft_magnitude': 'log', 'fft_window': 'hann', 'fft_drop_dc': False, 'n_train': 2420, 'n_val': 340, 'n_test': 2604, 'train_class_counts': [484, 484, 484, 484, 484, 0], 'test_class_counts': [651, 651, 651, 0, 0, 651]}
model params: 458,054  seq_len_out=15  lstm_input=384
forward ok: input (32, 1, 31, 6) -> logits (32, 6)
  [smoke] epoch   1 | train_loss 1.0640 | val_loss 1.0150 | val_acc 0.4853 | val_f1m 0.3191 | 1.0s  *
  [smoke] epoch   2 | train_loss 0.8846 | val_loss 0.8474 | val_acc 0.5412 | val_f1m 0.3920 | 0.4s  *
SMOKE OK | best_epoch=2 test_acc=0.3245 test_f1_macro=0.1863


## 3. Full LODO — FREQUENCY, run2 recipe (all 6 folds)
Defaults already encode the run2 recipe; flags shown explicitly for clarity.
Writes to `results/run2/frequency/`.

In [3]:
!cd "$PROJECT_PATH" && python -m experiments.run_lodo \
    --run_name run2 --input_view frequency \
    --fft_magnitude log --fft_window hann \
    --model_selection_metric val_acc --max_epochs 100 --patience 20

DeepConvLSTM  |  LODO on DAGHAR (standardized_view)
device       : cuda
data_path    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
results_dir  : /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/Project_ConvNet_DAGHAR/results/run2/frequency
run_name     : run2
input_view   : frequency  (seq_len=31, normalize=zscore_feature)
fft          : magnitude=log, window=hann, drop_dc=False
selection    : val_acc (max_epochs=100, patience=20)
channels(6): ['accel-x', 'accel-y', 'accel-z', 'gyro-x', 'gyro-y', 'gyro-z']
classes (6) : ['sit', 'stand', 'walk', 'stair up', 'stair down', 'run']
folds        : ['KuHar', 'MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']

#### LODO fold | held-out target = KuHar ####
  sources : ['MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']
  sizes   : train=35396 val=5412 test=144
  train class counts: [6709, 6709, 6709, 4522, 4522, 6225]
  test  class counts: [24, 24, 24, 24, 24, 24]
  [KuHar] epoch   1 | tr

## 4. (Optional) Full LODO — TIME, run2 recipe
Time view with the improved val-accuracy selection (for a clean time-vs-freq comparison under the same recipe). Writes to `results/run2/time/`.

In [4]:
!cd "$PROJECT_PATH" && python -m experiments.run_lodo \
    --run_name run2 --input_view time \
    --model_selection_metric val_acc --max_epochs 100 --patience 20

DeepConvLSTM  |  LODO on DAGHAR (standardized_view)
device       : cuda
data_path    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
results_dir  : /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/Project_ConvNet_DAGHAR/results/run2/time
run_name     : run2
input_view   : time  (seq_len=60, normalize=none)
selection    : val_acc (max_epochs=100, patience=20)
channels(6): ['accel-x', 'accel-y', 'accel-z', 'gyro-x', 'gyro-y', 'gyro-z']
classes (6) : ['sit', 'stand', 'walk', 'stair up', 'stair down', 'run']
folds        : ['KuHar', 'MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']

#### LODO fold | held-out target = KuHar ####
  sources : ['MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']
  sizes   : train=35396 val=5412 test=144
  train class counts: [6709, 6709, 6709, 4522, 4522, 6225]
  test  class counts: [24, 24, 24, 24, 24, 24]
  [KuHar] epoch   1 | train_loss 0.8352 | val_loss 0.6798 | val_acc 0.7411 | val_f1m 0.7402 | 5.4s  *

## 5. Compare against DAGHAR's reported ConvNet numbers

In [5]:
import json, os
RUN_NAME = 'run2'

# DAGHAR Table 9 — ConvNet[13] LODO accuracy (per held-out dataset).
DAGHAR = {
    'time':      {'KuHar':63.9,'MotionSense':65.6,'RealWorld_thigh':47.3,'RealWorld_waist':61.2,'UCI':70.1,'WISDM':53.9,'Mean':60.3},
    'frequency': {'KuHar':69.7,'MotionSense':85.6,'RealWorld_thigh':75.0,'RealWorld_waist':71.3,'UCI':81.8,'WISDM':79.1,'Mean':77.1},
}

def summarize(view, run_name=RUN_NAME):
    """Read all fold results.json (robust even if lodo_summary was overwritten)."""
    base = os.path.join(PROJECT_PATH, 'results', run_name, view)
    if not os.path.isdir(base):
        return None
    rows = {}
    for d in sorted(os.listdir(base)):
        fp = os.path.join(base, d, 'results.json')
        if d.startswith('fold_') and os.path.exists(fp):
            rows[d.replace('fold_','')] = json.load(open(fp))['metrics']['accuracy']*100
    return rows

for view in ['frequency', 'time']:
    rows = summarize(view)
    if not rows:
        print(f'\n[{view}] no results in results/{RUN_NAME}/{view} yet.'); continue
    print(f'\n==== {view.upper()} view (run {RUN_NAME}) ====')
    print(f"{'dataset':16s} {'ours':>7s} {'DAGHAR':>7s} {'diff':>7s}")
    accs=[]
    for tgt in ['KuHar','MotionSense','RealWorld_thigh','RealWorld_waist','UCI','WISDM']:
        if tgt not in rows: continue
        ours=rows[tgt]; accs.append(ours); ref=DAGHAR[view][tgt]
        print(f"{tgt:16s} {ours:6.1f}% {ref:6.1f}% {ours-ref:+7.1f}")
    mean=sum(accs)/len(accs)
    print(f"{'MEAN':16s} {mean:6.1f}% {DAGHAR[view]['Mean']:6.1f}% {mean-DAGHAR[view]['Mean']:+7.1f}")


==== FREQUENCY view (run run2) ====
dataset             ours  DAGHAR    diff
KuHar              59.0%   69.7%   -10.7
MotionSense        84.1%   85.6%    -1.5
RealWorld_thigh    70.6%   75.0%    -4.4
RealWorld_waist    67.9%   71.3%    -3.4
UCI                77.1%   81.8%    -4.7
WISDM              73.2%   79.1%    -5.9
MEAN               72.0%   77.1%    -5.1

==== TIME view (run run2) ====
dataset             ours  DAGHAR    diff
KuHar              61.8%   63.9%    -2.1
MotionSense        70.3%   65.6%    +4.7
RealWorld_thigh    63.5%   47.3%   +16.2
RealWorld_waist    62.3%   61.2%    +1.1
UCI                76.1%   70.1%    +6.0
WISDM              64.8%   53.9%   +10.9
MEAN               66.5%   60.3%    +6.2


## 6. (Optional) ablations to chase the last point or two
Each writes to its own run folder so nothing is overwritten. Re-run cell 5 with `RUN_NAME` set to the one you want to inspect.

In [6]:
# A) drop the DC bin
!cd "$PROJECT_PATH" && python -m experiments.run_lodo --run_name run2_dropdc --input_view frequency --fft_magnitude log --fft_window hann --fft_drop_dc --model_selection_metric val_acc
# B) macro-F1 model selection
!cd "$PROJECT_PATH" && python -m experiments.run_lodo --run_name run2_f1 --input_view frequency --fft_magnitude log --fft_window hann --model_selection_metric val_f1_macro
# C) no Hann window (pure log magnitude)
!cd "$PROJECT_PATH" && python -m experiments.run_lodo --run_name run2_nohann --input_view frequency --fft_magnitude log --fft_window none --model_selection_metric val_acc

DeepConvLSTM  |  LODO on DAGHAR (standardized_view)
device       : cuda
data_path    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
results_dir  : /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/Project_ConvNet_DAGHAR/results/run2_dropdc/frequency
run_name     : run2_dropdc
input_view   : frequency  (seq_len=30, normalize=zscore_feature)
fft          : magnitude=log, window=hann, drop_dc=True
selection    : val_acc (max_epochs=100, patience=20)
channels(6): ['accel-x', 'accel-y', 'accel-z', 'gyro-x', 'gyro-y', 'gyro-z']
classes (6) : ['sit', 'stand', 'walk', 'stair up', 'stair down', 'run']
folds        : ['KuHar', 'MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']

#### LODO fold | held-out target = KuHar ####
  sources : ['MotionSense', 'RealWorld_thigh', 'RealWorld_waist', 'UCI', 'WISDM']
  sizes   : train=35396 val=5412 test=144
  train class counts: [6709, 6709, 6709, 4522, 4522, 6225]
  test  class counts: [24, 24, 24, 24, 24, 24]
  [KuHar] e